## Step 1: Baseline feature and target table

The baselines use engineered features, which already sit in Process.csv (one row per batch). The four targets are in Laboratory.csv. Here I join them, pick the targets, and drop anything that leaks the answer. Output: one clean table for LASSO and XGBoost.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE=Path(r"C:\Users\Arpit Joshua Elias\OneDrive\Desktop\Pharma datasets")
TRAJ=BASE/"Process"

proc=pd.read_csv(BASE/"Process.csv",sep=";")
lab=pd.read_csv(BASE/"Laboratory.csv",sep=";")

print("process",proc.shape)
print("lab",lab.shape)

process (1005, 35)
lab (1005, 55)


In [2]:
df=proc.merge(lab,on="batch",suffixes=("","_lab"))
print("merged",df.shape)
print("batches kept",df["batch"].nunique())

merged (1005, 89)
batches kept 1005


## Pick the four targets

These are the quality attributes the model predicts: dissolution, hardness, weight RSD, tensile strength. I pull them into y and confirm none are missing.

In [3]:
targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]
y=df[targets].copy()

print("y",y.shape)
print(y.isna().sum())

y (1005, 4)
dissolution_av     0
tbl_av_hardness    0
tbl_rsd_weight     0
fct_tensile        0
dtype: int64


## Drop the leakage, build X

Product identity (code, batch, strength, size) is removed because the audit showed code alone explains most of the hardness variance. Process.csv also carries Drug release columns, which are basically the dissolution answer, so those go too. What remains is the honest engineered feature set.

In [4]:
leak=["batch","code","code_lab","strength","size","start",
      "Drug release average (%)","Drug release min (%)"]

X=proc.drop(columns=[c for c in leak if c in proc.columns]).copy()

print("X",X.shape)
print("X columns:")
print(list(X.columns))

X (1005, 31)
X columns:
['tbl_speed_mean', 'tbl_speed_change', 'tbl_speed_0_duration', 'total_waste', 'startup_waste', 'weekend', 'fom_mean', 'fom_change', 'SREL_startup_mean', 'SREL_production_mean', 'SREL_production_max', 'main_CompForce mean', 'main_CompForce_sd', 'main_CompForce_median', 'pre_CompForce_mean', 'tbl_fill_mean', 'tbl_fill_sd', 'cyl_height_mean', 'stiffness_mean', 'stiffness_max', 'stiffness_min', 'ejection_mean', 'ejection_max', 'ejection_min', 'Startup_tbl_fill_maxDifference', 'Startup_main_CompForce_mean', 'Startup_tbl_fill_mean', 'Residual solvent', 'Total impurities', 'Impurity O', 'Impurity L']


## Encode weekend, final check

weekend is text (no/yes), which the models cannot read, so I map it to 0/1. Then I confirm every feature is numeric and nothing is missing.

In [5]:
X["weekend"]=X["weekend"].map({"no":0,"yes":1}).astype(int)

print("X",X.shape)
print("non-numeric cols:",list(X.select_dtypes(exclude="number").columns))
print("missing values:",int(X.isna().sum().sum()))

X (1005, 31)
non-numeric cols: []
missing values: 72


In [6]:
miss=X.isna().sum()
print(miss[miss>0])

Residual solvent    18
Total impurities    18
Impurity O          18
Impurity L          18
dtype: int64


## Drop the chemistry columns

The only missing values sit in four incoming-material chemistry columns (18 batches each). These are also the only features whose status as a valid input is ambiguous. Dropping them gives a clean process-only feature set: no gaps, no leakage doubt, and it matches the 27 features used in the feasibility test.

In [7]:
chem=["Residual solvent","Total impurities","Impurity O","Impurity L"]
X=X.drop(columns=[c for c in chem if c in X.columns])

print("X",X.shape)
print("missing values:",int(X.isna().sum().sum()))
print(list(X.columns))

X (1005, 27)
missing values: 0
['tbl_speed_mean', 'tbl_speed_change', 'tbl_speed_0_duration', 'total_waste', 'startup_waste', 'weekend', 'fom_mean', 'fom_change', 'SREL_startup_mean', 'SREL_production_mean', 'SREL_production_max', 'main_CompForce mean', 'main_CompForce_sd', 'main_CompForce_median', 'pre_CompForce_mean', 'tbl_fill_mean', 'tbl_fill_sd', 'cyl_height_mean', 'stiffness_mean', 'stiffness_max', 'stiffness_min', 'ejection_mean', 'ejection_max', 'ejection_min', 'Startup_tbl_fill_maxDifference', 'Startup_main_CompForce_mean', 'Startup_tbl_fill_mean']


## Keep code aside for grouping

Product code is not a feature (it leaks), but I need it later to make the leave-one-product-code-out splits. So I store it separately, never inside X.

In [8]:
groups=df["code"].copy()

print("X",X.shape)
print("y",y.shape)
print("groups",groups.shape,"unique codes",groups.nunique())

X (1005, 27)
y (1005, 4)
groups (1005,) unique codes 25


## Step 2: Baseline setup

I build the two baselines from the paper: LASSO and XGBoost. Each predicts all four targets. To keep it honest I cross-validate with GroupKFold on product code, so no product appears in both train and test. I also add a predict-the-mean dummy as the floor the models must beat.

In [9]:
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

gkf=GroupKFold(n_splits=5)

print("targets:",targets)
print("features:",X.shape[1])
print("cv: GroupKFold 5 splits on code")

targets: ['dissolution_av', 'tbl_av_hardness', 'tbl_rsd_weight', 'fct_tensile']
features: 27
cv: GroupKFold 5 splits on code


## Train and score the three models

For each target I run the dummy, LASSO and XGBoost through the 5 grouped folds and record RMSE on the held-out fold. LASSO needs scaling so it goes in a pipeline. The dummy is the floor: any real model must beat it.

In [10]:
import warnings
warnings.filterwarnings("ignore")

models={
"dummy":DummyRegressor(strategy="mean"),
"lasso":Pipeline([("sc",StandardScaler()),("m",Lasso(alpha=0.1,max_iter=10000))]),
"xgb":XGBRegressor(n_estimators=300,max_depth=4,learning_rate=0.05,random_state=42,verbosity=0)
}

results={}
for tgt in targets:
    yt=y[tgt].values
    results[tgt]={}
    for name,model in models.items():
        rmses=[]
        for tr,te in gkf.split(X,yt,groups=groups):
            model.fit(X.iloc[tr],yt[tr])
            pred=model.predict(X.iloc[te])
            rmses.append(np.sqrt(mean_squared_error(yt[te],pred)))
        results[tgt][name]=np.mean(rmses)
    print(tgt,"done")

print("all targets done")

dissolution_av done
tbl_av_hardness done
tbl_rsd_weight done
fct_tensile done
all targets done


## Results table: model vs baseline

I put the RMSE for all three models side by side, one row per target. Lower is better. The check is simple: LASSO and XGBoost should both beat the dummy.

In [11]:
table=pd.DataFrame(results).T
table=table[["dummy","lasso","xgb"]]
table=table.round(3)

print(table)

                  dummy  lasso    xgb
dissolution_av    3.512  3.119  3.147
tbl_av_hardness  11.439  6.619  5.004
tbl_rsd_weight    0.558  0.538  0.605
fct_tensile       0.419  0.343  0.255


## Save the results table

I write the comparison table to CSV so it can go into the report and the weekly deliverable.

In [13]:
table.to_csv(BASE/"baseline_results.csv")
print("saved to",BASE/"baseline_results.csv")
print(table)

saved to C:\Users\Arpit Joshua Elias\OneDrive\Desktop\Pharma datasets\baseline_results.csv
                  dummy  lasso    xgb
dissolution_av    3.512  3.119  3.147
tbl_av_hardness  11.439  6.619  5.004
tbl_rsd_weight    0.558  0.538  0.605
fct_tensile       0.419  0.343  0.255
